# 🦠 MicroNet — Phase 3: VGAE Training (Colab GPU)

Variational Graph Autoencoder for microbial interaction network reconstruction.

- **Encoder**: 2-layer GCN → (μ, log σ) latent vectors per node
- **Decoder**: Inner product → edge probability
- **Loss**: ELBO = BCE reconstruction + KL divergence
- **Extra head**: Interaction type classifier (gLV-labeled supervision)

**Prerequisites**: Run the Inference notebook first to produce:
- `spieceasi_adj_weighted.tsv`
- `classified_interactions.tsv`
- `clr_abundance_matrix.tsv`

**Runtime**: Set to **GPU** (Runtime → Change runtime type → T4 GPU)

In [ ]:
# ── Install PyTorch Geometric (Colab GPU) ─────────────────────────────────
import torch
TORCH_VERSION = torch.__version__.split('+')[0]
CUDA_VERSION  = torch.version.cuda or 'cpu'
print(f'PyTorch {TORCH_VERSION}, CUDA {CUDA_VERSION}')

# Install matching PyG wheels
!pip install -q torch-geometric
!pip install -q torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH_VERSION}+{CUDA_VERSION.replace('.','')}.html
!pip install -q scikit-learn matplotlib

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch_geometric.utils import negative_sampling
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.data import Data
import numpy as np
import pandas as pd
from pathlib import Path
import json
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, average_precision_score
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  CONFIGURATION
# ══════════════════════════════════════════════════════════════════════════

# Paths (edit to match your setup)
ABUNDANCE_PATH    = 'results/profiling/clr_abundance_matrix.tsv'
ADJACENCY_PATH    = 'results/inference/spieceasi_adj_weighted.tsv'
INTERACTIONS_PATH = 'results/inference/classified_interactions.tsv'  # Optional
FUNCTIONAL_PATH   = None  # Optional: HUMAnN3 pathway matrix
OUTDIR            = 'results/gnn'

# Hyperparameters
HIDDEN_DIM  = 128
LATENT_DIM  = 64
EPOCHS      = 300
LR          = 1e-3
BETA        = 1.0   # KL divergence weight
EVAL_EVERY  = 10

Path(OUTDIR).mkdir(parents=True, exist_ok=True)
print('Configuration set ✓')

---
## Model Architecture

In [ ]:
class GCNEncoder(nn.Module):
    """Two-layer GCN encoder → (mu, log_std) latent vectors."""
    def __init__(self, in_ch, hidden_ch, out_ch, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(in_ch, hidden_ch)
        self.conv_mu = GCNConv(hidden_ch, out_ch)
        self.conv_logstd = GCNConv(hidden_ch, out_ch)
        self.dropout = nn.Dropout(dropout)
        self.bn = nn.BatchNorm1d(hidden_ch)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = self.bn(x)
        x = F.elu(x)
        x = self.dropout(x)
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)


class VGAE(nn.Module):
    """Variational Graph Autoencoder (Kipf & Welling, 2016)."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder

    def reparameterize(self, mu, log_std):
        if self.training:
            return mu + torch.randn_like(log_std) * torch.exp(log_std)
        return mu

    def encode(self, x, edge_index):
        self.mu, self.log_std = self.encoder(x, edge_index)
        return self.reparameterize(self.mu, self.log_std)

    def decode(self, z, edge_index):
        src, dst = edge_index
        return (z[src] * z[dst]).sum(dim=-1)

    def decode_all(self, z):
        return torch.sigmoid(z @ z.T)

    def recon_loss(self, z, pos_edge_index, neg_edge_index=None):
        pos_loss = -F.logsigmoid(self.decode(z, pos_edge_index)).mean()
        if neg_edge_index is None:
            neg_edge_index = negative_sampling(
                pos_edge_index, num_nodes=z.size(0),
                num_neg_samples=pos_edge_index.size(1))
        neg_loss = -F.logsigmoid(-self.decode(z, neg_edge_index)).mean()
        return pos_loss + neg_loss

    def kl_loss(self):
        return -0.5 * torch.mean(
            1 + 2 * self.log_std - self.mu.pow(2) - (2 * self.log_std).exp())

    def elbo(self, z, pos_edge_index, neg_edge_index=None, beta=1.0):
        return self.recon_loss(z, pos_edge_index, neg_edge_index) + beta * self.kl_loss()


class InteractionTypeHead(nn.Module):
    """MLP for edge-type classification using gLV labels as supervision."""
    TYPES = ['Mutualism', 'Competition', 'Parasitism', 'Commensalism', 'Amensalism']

    def __init__(self, latent_dim, hidden=64):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(latent_dim * 2, hidden), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(hidden, len(self.TYPES)))

    def forward(self, z, edge_index):
        src, dst = edge_index
        return self.mlp(torch.cat([z[src], z[dst]], dim=-1))

print('Models defined ✓')

---
## Data Preparation

In [ ]:
def build_graph_data(abundance_path, adj_path, functional_path=None,
                     interaction_labels_path=None):
    """Build PyG Data from CLR matrix + SPIEC-EASI adjacency."""
    clr = pd.read_csv(abundance_path, sep='\t', index_col=0)
    taxa = clr.columns.tolist()
    node_features = clr.T.values.astype(np.float32)  # N × S

    if functional_path:
        func = pd.read_csv(functional_path, sep='\t', index_col=0)
        func = func.reindex(taxa).fillna(0)
        node_features = np.hstack([node_features, func.values.astype(np.float32)])

    adj_df = pd.read_csv(adj_path, sep='\t', index_col=0)
    adj_df = adj_df.reindex(index=taxa, columns=taxa).fillna(0)
    adj = adj_df.values.astype(np.float32)

    rows, cols = np.where(adj != 0)
    edge_index = torch.tensor(np.stack([rows, cols]), dtype=torch.long)
    edge_weight = torch.tensor(adj[rows, cols], dtype=torch.float)
    x = torch.tensor(node_features, dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_weight,
                num_nodes=len(taxa))
    data.taxa_names = taxa

    if interaction_labels_path:
        try:
            labels_df = pd.read_csv(interaction_labels_path, sep='\t')
            type_map = {t: i for i, t in enumerate(InteractionTypeHead.TYPES)}
            taxa_idx = {t: i for i, t in enumerate(taxa)}
            labeled_edges, labeled_types = [], []
            for _, row in labels_df.iterrows():
                i = taxa_idx.get(row.get('taxon_i_name'))
                j = taxa_idx.get(row.get('taxon_j_name'))
                t = type_map.get(row.get('interaction_type'), 4)
                if i is not None and j is not None:
                    labeled_edges.append([i, j])
                    labeled_types.append(t)
            if labeled_edges:
                data.labeled_edge_index = torch.tensor(labeled_edges, dtype=torch.long).T
                data.edge_labels = torch.tensor(labeled_types, dtype=torch.long)
                print(f'  Loaded {len(labeled_edges)} labeled edges for supervision')
        except FileNotFoundError:
            print('  No interaction labels file found — unsupervised mode')

    return data

# Build graph
print('Building graph data ...')
data = build_graph_data(ABUNDANCE_PATH, ADJACENCY_PATH,
                        FUNCTIONAL_PATH, INTERACTIONS_PATH)
print(f'Graph: {data.num_nodes} nodes, {data.edge_index.shape[1]//2} undirected edges')
print(f'Node features: {data.x.shape[1]} dimensions')

In [ ]:
# ── Train/Val/Test split using RandomLinkSplit ──────────────────────────────
transform = RandomLinkSplit(
    num_val=0.1, num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=False,
)
train_data, val_data, test_data = transform(data)

# Attach labeled edges to train_data for the classification head
if hasattr(data, 'labeled_edge_index'):
    train_data.labeled_edge_index = data.labeled_edge_index
    train_data.edge_labels = data.edge_labels

print(f'Train edges: {train_data.edge_index.shape[1]}')
print(f'Val supervision edges: {val_data.edge_label_index.shape[1]}')
print(f'Test supervision edges: {test_data.edge_label_index.shape[1]}')

---
## Training Loop (GPU-accelerated)

In [ ]:
def train_epoch(model, head, data, optimizer, device, beta=1.0):
    model.train()
    optimizer.zero_grad()
    x = data.x.to(device)
    edge_index = data.edge_index.to(device)  # RandomLinkSplit stores train edges here
    z = model.encode(x, edge_index)
    loss = model.elbo(z, edge_index, beta=beta)

    if hasattr(data, 'labeled_edge_index'):
        le = data.labeled_edge_index.to(device)
        logits = head(z, le)
        labels = data.edge_labels.to(device)
        loss = loss + 0.5 * F.cross_entropy(logits, labels)

    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, val_data, device):
    model.eval()
    x = val_data.x.to(device)
    edge_index = val_data.edge_index.to(device)
    z = model.encode(x, edge_index)

    pos_edge = val_data.edge_label_index[:, val_data.edge_label == 1].to(device)
    neg_edge = val_data.edge_label_index[:, val_data.edge_label == 0].to(device)

    pos_pred = torch.sigmoid(model.decode(z, pos_edge)).cpu().numpy()
    neg_pred = torch.sigmoid(model.decode(z, neg_edge)).cpu().numpy()

    y_true = np.concatenate([np.ones(len(pos_pred)), np.zeros(len(neg_pred))])
    y_pred = np.concatenate([pos_pred, neg_pred])

    auc = roc_auc_score(y_true, y_pred)
    ap  = average_precision_score(y_true, y_pred)
    return auc, ap, z.cpu()

print('Training functions defined ✓')

In [ ]:
# ── Initialize model & optimizer ───────────────────────────────────────────
encoder = GCNEncoder(data.x.shape[1], HIDDEN_DIM, LATENT_DIM)
model = VGAE(encoder).to(device)
head  = InteractionTypeHead(LATENT_DIM).to(device)

# Only include head params when labeled edges exist
params = list(model.parameters())
if hasattr(train_data, 'labeled_edge_index'):
    params += list(head.parameters())
optimizer = torch.optim.Adam(params, lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {total_params:,}')
print(f'Training on: {device}')

In [ ]:
# ── Train! ─────────────────────────────────────────────────────────────────
history = {'loss': [], 'val_auc': [], 'val_ap': []}
best_auc = 0
best_z = None

print(f'Training VGAE for {EPOCHS} epochs ...')
print(f'{"Epoch":>6} {"Loss":>10} {"AUC":>10} {"AP":>10}')
print('-' * 40)

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, head, train_data, optimizer, device, beta=BETA)
    scheduler.step()
    history['loss'].append(loss)

    if epoch % EVAL_EVERY == 0:
        auc, ap, z = evaluate(model, val_data, device)
        history['val_auc'].append(auc)
        history['val_ap'].append(ap)
        marker = ''
        if auc > best_auc:
            best_auc = auc
            best_z = z
            torch.save(model.state_dict(), f'{OUTDIR}/best_vgae.pt')
            np.save(f'{OUTDIR}/best_embeddings.npy', z.numpy())
            marker = ' ★ best'
        print(f'{epoch:>6d} {loss:>10.4f} {auc:>10.4f} {ap:>10.4f}{marker}')

print(f'\n✅ Training complete. Best validation AUC: {best_auc:.4f}')

In [ ]:
# ── Learning curves ────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history['loss'], alpha=0.7)
ax1.set_title('ELBO Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_yscale('log')

eval_epochs = list(range(EVAL_EVERY, EPOCHS+1, EVAL_EVERY))
ax2.plot(eval_epochs, history['val_auc'], label='AUC', marker='o', ms=3)
ax2.plot(eval_epochs, history['val_ap'],  label='AP',  marker='s', ms=3)
ax2.set_title('Validation Metrics'); ax2.set_xlabel('Epoch'); ax2.legend()
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(f'{OUTDIR}/training_curves.png', dpi=150)
plt.show()

---
## Final Evaluation & Export

In [ ]:
# ── Test set evaluation ────────────────────────────────────────────────────
model.load_state_dict(torch.load(f'{OUTDIR}/best_vgae.pt', weights_only=True))
test_auc, test_ap, _ = evaluate(model, test_data, device)
print(f'Test AUC: {test_auc:.4f}')
print(f'Test AP:  {test_ap:.4f}')

In [ ]:
# ── Reconstruct full edge probability matrix ───────────────────────────────
model.eval()
with torch.no_grad():
    z = model.encode(train_data.x.to(device), train_data.edge_index.to(device))
    z = z.cpu()

prob_matrix = torch.sigmoid(z @ z.T).numpy()
prob_df = pd.DataFrame(prob_matrix, index=data.taxa_names, columns=data.taxa_names)
prob_df.to_csv(f'{OUTDIR}/predicted_edge_probabilities.tsv', sep='\t')

# Save training history
with open(f'{OUTDIR}/training_history.json', 'w') as f:
    json.dump(history, f)

print(f'\n✅ All outputs saved to {OUTDIR}/')
print(f'  - best_vgae.pt (model weights)')
print(f'  - best_embeddings.npy (latent node embeddings)')
print(f'  - predicted_edge_probabilities.tsv')
print(f'  - training_history.json')
print(f'  - training_curves.png')

In [ ]:
# ── Predicted edge heatmap (top taxa) ─────────────────────────────────────
n_show = min(40, len(data.taxa_names))
fig, ax = plt.subplots(figsize=(12, 10))
sns_data = prob_df.iloc[:n_show, :n_show]

import seaborn as sns
sns.heatmap(sns_data, cmap='viridis', vmin=0, vmax=1,
            xticklabels=True, yticklabels=True,
            ax=ax, square=True, linewidths=0.2)
ax.set_title(f'VGAE Predicted Edge Probabilities (top {n_show} taxa)', fontsize=13)
ax.tick_params(labelsize=6)
plt.tight_layout()
plt.savefig(f'{OUTDIR}/predicted_edges_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary stats
high_conf = (prob_matrix > 0.7).sum() // 2
total_possible = len(data.taxa_names) * (len(data.taxa_names) - 1) // 2
print(f'\nHigh-confidence predicted edges (p > 0.7): {high_conf} / {total_possible}')